In [1]:
import torch
from torch import nn, optim
import numpy as np
import requests, gzip, os, hashlib, numpy
import matplotlib.pyplot as plt

In [11]:
def fetch(url):
    fp = os.path.join("/tmp", hashlib.md5(url.encode('utf-8')).hexdigest())
    if os.path.isfile(fp):
        with open(fp, "rb") as f:
            dat = f.read()
    else:
        with open(fp, "wb") as f:
            dat = requests.get(url).content
            f.write(dat)
    return np.frombuffer(gzip.decompress(dat), dtype=np.uint8).copy()

In [19]:
X_train = fetch("http://yann.lecun.com/exdb/mnist/train-images-idx3-ubyte.gz")[0x10:].reshape((-1, 28, 28))
Y_train = fetch("http://yann.lecun.com/exdb/mnist/train-labels-idx1-ubyte.gz")[8:]
X_test = fetch("http://yann.lecun.com/exdb/mnist/t10k-images-idx3-ubyte.gz")[0x10:].reshape((-1, 28, 28))
Y_test = fetch("http://yann.lecun.com/exdb/mnist/t10k-labels-idx1-ubyte.gz")[8:]

In [20]:
X_train = torch.tensor(X_train).float()
Y_train = torch.tensor(Y_train)
X_test = torch.tensor(X_test).float()
Y_test = torch.tensor(Y_test)


In [21]:
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.layer1 = nn.Linear(784, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 10)
        
    def forward(self, x):
        x = nn.functional.relu(self.layer1(x))
        x = nn.functional.relu(self.layer2(x))
        x = nn.functional.log_softmax(self.layer3(x), dim=1)
        
        return x
        

In [22]:
model = Model()

loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [23]:
# first model
for i in range(100):
    optimizer.zero_grad()
    y_hat = model(X_train.view(-1, 784))
    
    loss = loss_function(y_hat,Y_train)
    loss.backward()
    optimizer.step()
    
    print(f'{str(i)}: loss: {loss.item()}')

0: loss: 12.470746040344238
1: loss: 54.077354431152344
2: loss: 214.89952087402344
3: loss: 13.599536895751953
4: loss: 4.158182621002197
5: loss: 2.404599666595459
6: loss: 2.315293550491333
7: loss: 2.2702925205230713
8: loss: 2.2146339416503906
9: loss: 2.1498873233795166
10: loss: 2.0910255908966064
11: loss: 2.0455124378204346
12: loss: 2.00924015045166
13: loss: 1.9784672260284424
14: loss: 1.9508765935897827
15: loss: 1.9258626699447632
16: loss: 1.9029337167739868
17: loss: 1.8811143636703491
18: loss: 1.8597899675369263
19: loss: 1.8385272026062012
20: loss: 1.8174707889556885
21: loss: 1.7952758073806763
22: loss: 1.7720741033554077
23: loss: 1.7460334300994873
24: loss: 1.7169963121414185
25: loss: 1.6822165250778198
26: loss: 1.6433290243148804
27: loss: 1.610426425933838
28: loss: 1.6221463680267334
29: loss: 1.8455930948257446
30: loss: 1.856019139289856
31: loss: 1.7657208442687988
32: loss: 1.695442795753479
33: loss: 1.6845817565917969
34: loss: 1.6815677881240845
35: